In [1]:
using Revise
using Pkg;
#Pkg.develop(path="/home/gert/Projects/FusionRings.jl/")
#Pkg.develop(path="/Users/gertvercleyen/Projects/FusionRings.jl/")
using FusionRings
using Oscar
using JSON
using Base.Threads


Welcome to Nemo version 0.52.4

Nemo comes with absolutely no warranty whatsoever
 ┌───────┐   GAP 4.15.1 of 2025-10-18
 │  GAP  │   https://www.gap-system.org
 └───────┘   Architecture: x86_64-pc-linux-gnu-julia1.12-64-kv10
 Configuration:  gmp 6.3.0, Julia GC, Julia 1.12.2, readline
 Loading the library and packages ...
 Packages:   AClib 1.3.3, Alnuth 3.2.1, AtlasRep 2.1.9, AutoDoc 2025.10.16, 
             AutPGrp 1.11.1, Browse 1.8.21, CaratInterface 2.3.7, CRISP 1.4.8, 
             Cryst 4.1.30, CrystCat 1.1.10, CTblLib 1.3.11, 
             curlInterface 2.4.2, FactInt 1.6.3, FGA 1.5.0, Forms 1.2.13, 
             GAPDoc 1.6.7, genss 1.6.9, IO 4.9.3, IRREDSOL 1.4.4, 
             JuliaInterface 0.16.2, LAGUNA 3.9.7, orb 5.0.1, 
             PackageManager 1.6.3, Polenta 1.3.11, Polycyclic 2.17, 
             PrimGrp 4.0.1, RadiRoot 2.9, recog 1.4.4, ResClasses 4.7.4, 
             SmallGrp 1.5.4, Sophus 1.27, SpinSym 1.5.2, StandardFF 1.0, 
             TomLib 1.2.11, TransGrp

In [2]:
function samefield_characters( r )
	to_composite_field( characters( r ), simplify_field = true )
end

samefield_characters (generic function with 1 method)

In [4]:
function export_characters( i::Int ) 
	fn1 = "/home/gert/Tests/characters/chars_"* string(i) *".mrdi"
	fn2 = "/home/gert/Tests/characters/chars_injection_"* string(i) *".mrdi"

	function export_new_chars(i) 
		try 
			chars, f  = samefield_characters(frl[i])
			generator = gen( parent( chars[1] ) )
			Oscar.save( fn1, chars )
			Oscar.save( fn2, f(generator) )
		catch e2
			Oscar.save( fn1, ZZ.( [ 0 ] ) )
			Oscar.save( fn2, ZZ.( [ 0 ] ) )
		end	
	end
	
	try
		chars = Oscar.load(fn1)
		f     = Oscar.load(fn2)
		if chars == ZZ.([0])
			export_new_chars(i)
		end
	catch e
		export_new_chars(i)
	end
end

export_characters (generic function with 1 method)

In [5]:
function create_ind()
	indices = []
	fn(i) = "/home/gert/Tests/characters/chars_injection_"* string(i) *".mrdi"
	for j in 1:352
        if !FusionRings.is_commutative(frl[j])
            continue
        else
    		try 
    			f = Oscar.load(fn(j))
    			if f == ZZ.([0])
    				push!( indices, j )
    			end
    			continue
    		catch e
    			push!( indices, j )
    		end
        end
	end
	indices
end

create_ind (generic function with 1 method)

In [6]:
reverse(create_ind())

15-element Vector{Any}:
 346
 300
 282
 281
 273
 254
 241
 229
 228
 201
 181
 180
 149
 107
  88

In [ ]:
@threads for i in create_ind()
    export_characters(i)
    print(" * ")
end

2×2 Matrix{QQBarFieldElem}:
 {a1: 1.00}  {a1: 1.00}
 {a1: 1.00}  {a1: -1.00}

In [9]:
qqbar(1)

{a1: 1.00000}

# Finding Characters By Solving System of equations

In [55]:
function solve_character_equations( ring )
    r    = rank(ring)
    R, χ = polynomial_ring( QQ, :χ => 1:r )  
    m    = multiplication_table(ring) 
    setunit( pol ) = evaluate( pol, [χ[1]], [R(1)] )
    I = 
        ideal( 
            R, 
            unique(
            vec(
                [ 
                    setunit( χ[i]*χ[j] - sum( m[i,j,k]*χ[k] for k in 1:r ) ) 
                    for i in 2:r, j in 2:r 
                ]
            ))
        )
    groebner_basis(I,complete_reduction = true)
end

solve_character_equations (generic function with 1 method)

In [57]:
solve_character_equations( frl[300] )

Gröbner basis with elements
  1: χ[9]^2 - χ[2] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  2: χ[8]*χ[9] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9] - 1
  3: χ[7]*χ[9] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  4: χ[6]*χ[9] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  5: χ[5]*χ[9] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  6: χ[4]*χ[9] - χ[6] - χ[7] - χ[8] - χ[9]
  7: χ[3]*χ[9] - χ[7] - χ[8] - χ[9]
  8: χ[2]*χ[9] - χ[8]
  9: χ[8]^2 - χ[2] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  10: χ[7]*χ[8] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  11: χ[6]*χ[8] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  12: χ[5]*χ[8] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  13: χ[4]*χ[8] - χ[6] - χ[7] - χ[8] - χ[9]
  14: χ[3]*χ[8] - χ[7] - χ[8] - χ[9]
  15: χ[2]*χ[8] - χ[9]
  16: χ[7]^2 - χ[2] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9] - 1
  17: χ[6]*χ[7] - χ[3] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  18: χ[5]*χ[7] - χ[4] - χ[6] - χ[7] - χ[8] - χ[9]
  19: χ[4]*χ[7] - χ[5] - χ[7] - χ[8] - χ[9]

In [8]:
frl[3]

LoadError: UndefVarError: `frl` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Finding characters via eigenvalues

In [26]:
function evchars( ring )
    mt = multiplication_table( ring )
    r  = rank( ring )
    espaces = [ eigenvalues( matrix( algebraic_closure(QQ), mt[i,:,:] ) ) for i in 1:r ]
    
end

evchars (generic function with 1 method)

In [27]:
evchars( frl[ 13 ] )

4-element Vector{Vector{QQBarFieldElem}}:
 [{a1: 1.00000}]
 [{a3: 1.87939}, {a1: 1.00000}, {a3: -0.347296}, {a3: -1.53209}]
 [{a3: 2.53209}, {a3: 1.34730}, {a3: -0.879385}, {a1: 0}]
 [{a3: 2.87939}, {a3: 0.652704}, {a3: -0.532089}, {a1: -1.00000}]

# Finding characters by indivivdual diagonalization

In [55]:
# Quick test for hypothesis that each fusion ring has 1 mat that has r different evals
function chars( ring )
    mt = multiplication_table( ring )
    r  = rank( ring )
    espaces = [ eigenspaces( matrix( algebraic_closure(QQ), mt[i,:,:] ) ) for i in 2:r ]
    
end

chars (generic function with 1 method)

In [56]:
t = chars( frl[12] )

3-element Vector{Dict{QQBarFieldElem, AbstractAlgebra.Generic.MatSpaceElem{QQBarFieldElem}}}:
 Dict({a2: 1.61803} => [{a2: 0.618034} {a1: 1.00000} {a1: 0} {a1: 0}; {a1: 0} {a1: 0} {a2: 0.618034} {a1: 1.00000}], {a2: -0.618034} => [{a2: -1.61803} {a1: 1.00000} {a1: 0} {a1: 0}; {a1: 0} {a1: 0} {a2: -1.61803} {a1: 1.00000}])
 Dict({a2: 1.61803} => [{a2: 0.618034} {a1: 0} {a1: 1.00000} {a1: 0}; {a1: 0} {a2: 0.618034} {a1: 0} {a1: 1.00000}], {a2: -0.618034} => [{a2: -1.61803} {a1: 0} {a1: 1.00000} {a1: 0}; {a1: 0} {a2: -1.61803} {a1: 0} {a1: 1.00000}])
 Dict({a1: -1.00000} => [{a1: 0} {a1: -1.00000} {a1: 1.00000} {a1: 0}; {a1: -1.00000} {a1: -1.00000} {a1: 0} {a1: 1.00000}], {a2: 2.61803} => [{a2: 0.381966} {a2: 0.618034} {a2: 0.618034} {a1: 1.00000}], {a2: 0.381966} => [{a2: 2.61803} {a2: -1.61803} {a2: -1.61803} {a1: 1.00000}])

In [52]:
function normalize_eigenspaces( espace )
    mynorm( v ) = sqrt( sum( v[i]^2 for i in 1:length(v) ) )
    normalize( space ) = [ space[i,:] ./ mynorm( space[i,:] ) for i in 1:size(space,1) ]
    Dict( k => normalize( s ) for ( k, s ) in espace )
end

normalize_eigenspaces (generic function with 1 method)

In [58]:
map( collect ∘ normalize_eigenspaces, t ) 

3-element Vector{Vector{Pair{QQBarFieldElem, Vector{Vector{QQBarFieldElem}}}}}:
 [{a2: 1.61803} => [[{a4: 0.525731}, {a4: 0.850651}, {a1: 0}, {a1: 0}], [{a1: 0}, {a1: 0}, {a4: 0.525731}, {a4: 0.850651}]], {a2: -0.618034} => [[{a4: -0.850651}, {a4: 0.525731}, {a1: 0}, {a1: 0}], [{a1: 0}, {a1: 0}, {a4: -0.850651}, {a4: 0.525731}]]]
 [{a2: 1.61803} => [[{a4: 0.525731}, {a1: 0}, {a4: 0.850651}, {a1: 0}], [{a1: 0}, {a4: 0.525731}, {a1: 0}, {a4: 0.850651}]], {a2: -0.618034} => [[{a4: -0.850651}, {a1: 0}, {a4: 0.525731}, {a1: 0}], [{a1: 0}, {a4: -0.850651}, {a1: 0}, {a4: 0.525731}]]]
 [{a1: -1.00000} => [[{a1: 0}, {a2: -0.707107}, {a2: 0.707107}, {a1: 0}], [{a2: -0.577350}, {a2: -0.577350}, {a1: 0}, {a2: 0.577350}]], {a2: 2.61803} => [[{a2: 0.276393}, {a2: 0.447214}, {a2: 0.447214}, {a2: 0.723607}]], {a2: 0.381966} => [[{a2: 0.723607}, {a2: -0.447214}, {a2: -0.447214}, {a2: 0.276393}]]]

In [44]:
size(test,2)

4

In [53]:
collect( normalize_eigenspaces( t[1] ) )

2-element Vector{Pair{QQBarFieldElem, Vector{Vector{QQBarFieldElem}}}}:
   {a2: 1.61803} => [[{a4: 0.525731}, {a4: 0.850651}, {a1: 0}, {a1: 0}], [{a1: 0}, {a1: 0}, {a4: 0.525731}, {a4: 0.850651}]]
 {a2: -0.618034} => [[{a4: -0.850651}, {a4: 0.525731}, {a1: 0}, {a1: 0}], [{a1: 0}, {a1: 0}, {a4: -0.850651}, {a4: 0.525731}]]

In [48]:
collect( t[1] )

2-element Vector{Pair{QQBarFieldElem, AbstractAlgebra.Generic.MatSpaceElem{QQBarFieldElem}}}:
   {a2: 1.61803} => [{a2: 0.618034} {a1: 1.00000} {a1: 0} {a1: 0}; {a1: 0} {a1: 0} {a2: 0.618034} {a1: 1.00000}]
 {a2: -0.618034} => [{a2: -1.61803} {a1: 1.00000} {a1: 0} {a1: 0}; {a1: 0} {a1: 0} {a2: -1.61803} {a1: 1.00000}]

# Claus' solution

QQBarFieldElem

In [2]:
module GertMod

using Oscar



function (a::QQBarField)(b::AbsSimpleNumFieldElem, v::AbsSimpleNumFieldEmbedding)

  f = minpoly(b)

  r = roots(a, f)

  val = v(b)

  for x = r

    if contains_zero(parent(val)(x) - val)

      return x

    end

  end

  error("no root found")

end



Oscar.minpoly(a::QQPolyRing, b::QQFieldElem) = gen(a)-b


function Gert(a::Vector{Matrix{ZZRingElem}})

  a = [ matrix(QQ, x) for x = a ]

  i = 1

  e = Hecke.common_eigenspaces(a)

  if length(e) == length(a)

    return e

  end
  # what does this do?
  x = gen(Hecke.Globals.Qx)
  # what does this do?
  Qx = parent(x)
  # what does this do?
  ee = Set([x-t for t = s] for s = keys(e))

  K = QQ

  while length(e) < length(a)

    f = charpoly(a[i])

    lf = factor(f)

    n = 0

    for (p, k) = lf

      if degree(p) > 1

        K = number_field(p)[1]

        e = Hecke.common_eigenspaces([map_entries(K, x) for x = a])

        eee = Set([minpoly(Qx, t) for t = x] for x = keys(e))

        if length(ee) == 0

          ee = eee

        else

          union!(ee, eee)

        end

      end

    end

    if sum(maximum(map(degree, x)) for x = ee; init = 0) == length(a)

      break

    end

    i += 1

  end



  ee = collect(ee)

  d = [[] for i=a]

  X = QQBarField()

  v = zero_matrix(X, 0, length(a))

  for x = ee

    f = argmax(degree, x)

    k, _ = number_field(f)

    em = []

    for i = infinite_places(k)

      if isreal(i)

        push!(em, embedding(i))

      else

        append!(em, embeddings(i))

      end

    end

    c = Hecke.common_eigenspaces([map_entries(k, x) for x= a])

    # keys(c) = vectors of evals (might have multiple) 
    for k = keys(c)

      #@show k

      mp = [minpoly(Qx, t) for t = k]

      #@show mp

      x != mp && continue

      for j = em

        v = vcat(v, map_entries(x->X(x, j), c[k]))

        for i = 1:length(a)

          push!(d[i], X(k[i], j))

        end

      end

      break

    end

  end

        

  return d, v



end



export Gert



end #GertMod



using .GertMod

In [3]:
function Gertchars( ring ) 
    is_commutative( ring ) || error("Noncom ring") 
    mt = multiplication_table( ring )
    matvec = [ ZZ.( mt[ i, :, : ] ) for i in 1:rank(ring) ]
    sort_mat( mat )  = sortslices( mat, dims = 1, by = char_sort_crit )
    chars = GertMod.Gert( matvec )
    chars[2] |> normalize_first_col |> sort_mat
end

function char_sort_crit( v )
	RR = ArbField(64);
	CC = AcbField(64);
	conv(x) = convert(Float64,x)
	# Abs values of elements of v
	absval(vec) = conv.( RR.(abs2.(vec)) )
	# Angles of elements of v
	angl(vec) = conv.( real.( log.( CC.( vec) ) ./ CC( 2 * pi * im ) ) )
			
	( Int( all(isreal.(v)) ), absval(v), angl(v) )
end


function normalize_first_col( mat )
    m, n = size( mat ) 
    
    [ mat[i,j]/mat[i,1] for i in 1:m, j in 1:n ]
end


normalize_first_col (generic function with 1 method)

In [4]:
function is_diagonalizing_matrix( mat, ring::FusionRing )
    qqb  = algebraic_closure(QQ)
	mt   = FusionRings.multiplication_table( ring )
	r    = FusionRings.rank(ring)
	mats = [ matrix( qqb, mt[ i, :, : ] ) for i ∈ 1:r ]
    mat  = matrix( qqb, mat )

  all( is_diagonal( mat * m * inv(mat) ) for m in mats )
end


is_diagonalizing_matrix (generic function with 1 method)

In [5]:
function testchars( i )
    try
        chrs = Gertchars( frl[i] )
        return is_diagonalizing_matrix( chrs, frl[i] )
    catch e 
        return missing
    end
end

testchars (generic function with 1 method)

In [7]:
tests = [ testchars( i ) for i in 1:40 ] 

40-element Vector{Union{Missing, Bool}}:
  missing
  missing
 1
 1
  missing
 1
 1
  missing
 1
 1
 1
 1
 1
 ⋮
 1
 1
 1
 1
 1
 1
  missing
 1
 1
 1
 1
 1